# AutoMage demo

Classify a georeferenced image using SAM3 and a default or custom dictionary. Inspect overlapping **features** and optional non-overlapping **superpixel objects**, along with their spatial relationships.

The notebook calls the AutoMage Python library. Select a GPU runtime in Colab. Local execution needs an environment with AutoMage and GPU PyTorch installed. Colab execution has not yet been verified.

SAM3 weights are included with AutoMage and loaded automatically. No separate model download or credentials are needed.

## Install and check the GPU

In Colab, the following cell installs AutoMage from GitHub. Locally, install the repository with `python -m pip install ".[notebook]"` before opening the notebook. The GitHub installation requires the repository contents to have been published.

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if shutil.which("git-lfs") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet",
        "automage[notebook] @ git+https://github.com/aboettcher-sig/automage.git",
    ], check=True)

import torch
from automage import Config, classify
from automage.raster import plan

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime or install PyTorch for your GPU.")
print(torch.__version__, torch.cuda.get_device_name(0))

## Image, dictionary, and settings

The included Boulder image is a 1600×1200 GeoTIFF crop from July 2023. Replace `INPUT` with another GeoTIFF path to use your own image. In Colab you can upload it through the Files panel or mount Drive.

Leave `DICTIONARY` unset to use the default 17-class dictionary. Set it to a JSON path for a custom dictionary. `TARGET_GSD_M=None` uses native resolution. Output rasters retain the source grid and CRS. Superpixel size is measured in source pixels.

Use a new output directory for each run. The default is a unique temporary directory outside the checkout. Save the resulting files before ending a Colab session.

In [ ]:
from importlib.resources import files
from datetime import datetime, timezone
import tempfile

INPUT = Path(str(files("automage").joinpath("data/boulder_features_objects.tif")))
DICTIONARY = None
OUTPUT = Path(tempfile.gettempdir()) / "automage-demo" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%f")

TARGET_GSD_M = None
OVERLAP = 0.20
PROMPT_BATCH = 8
AUTOCAST_DTYPE = None  # Optional "float16" on a compatible GPU.
CREATE_OBJECTS = True
OBJECT_REGION_PX = 28
OBJECT_COMPACTNESS = 12.0
WALL_SECONDS = 5400

config = Config(
    target_gsd=TARGET_GSD_M, overlap=OVERLAP,
    prompt_batch=PROMPT_BATCH, autocast_dtype=AUTOCAST_DTYPE,
    objects=CREATE_OBJECTS, object_region_px=OBJECT_REGION_PX,
    object_compactness=OBJECT_COMPACTNESS, wall_seconds=WALL_SECONDS,
)
if DICTIONARY is not None:
    config.dictionary = str(DICTIONARY)

preview = plan(INPUT, config)
print("Image:", INPUT)
print("Output:", OUTPUT)
print("Windows:", preview["window_count"])
print("Ground resolution (m/pixel):", preview["presented_ground_gsd_m"])

## Run classification

This cell processes the whole selected image using the same library as the command line. Inference requires the GPU; superpixel generation and exports use the CPU.

In [ ]:
summary = classify(INPUT, OUTPUT, config)
print("Status:", summary["state"])
print("Features:", summary["feature_count"])
print("Objects:", summary["objects"])

## Inspect results

- `classification.gpkg` contains features, optional superpixel objects, and their relationship table.
- `features/*.tif` contains score and feature ID bands for each detected class.
- `objects.tif` contains optional superpixel IDs matching the GeoPackage.
- `overview.png` displays imagery, features, and proposed object classes.
- `summary.json` records counts, settings, execution status, and provenance.

Open the GeoPackage in GIS software to review objects using `class_proposed`, `class_final`, `needs_review`, `reviewed`, and `notes`. Scores and review flags are not classification accuracy estimates.

The companion `.work` directory contains diagnostics and is required for verifying reuse of a complete result. Partial runs cannot be resumed.

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUTPUT / "overview.png")))
for path in sorted(OUTPUT.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT))